In [19]:
import sqlite3
import os
import shutil
con = sqlite3.connect("annotations.sqlite3")
cur = con.cursor()

In [20]:
buffer = dict()
differences = []

for row in cur.execute("SELECT * FROM decisions ORDER BY filename"):
    id_, username, filename, decision, ground_truth, correct, timestamp = row
    if not filename in buffer:
        buffer[filename] = [username, decision, ground_truth, correct]
    else:
        assert buffer[filename][0] != username # users should be different
        if buffer[filename][1] != decision:
            print(f"Difference for {filename}:")
            print(f"  {buffer[filename][0]}: {buffer[filename][1]} (correct={buffer[filename][3]})")
            print(f"  {username}: {decision} (correct={correct})")
            differences.append((filename, correct, buffer[filename][0], buffer[filename][1], username, decision))

print(f"Total differences: {len(differences)}")

Difference for with_signal/12769132_FM.png:
  admin: 1 (correct=1)
  vadim: 0 (correct=0)
Difference for with_signal/12776974_USB.png:
  admin: 1 (correct=1)
  vadim: 0 (correct=0)
Difference for with_signal/12784285_GMSK USP.png:
  admin: 0 (correct=0)
  vadim: 1 (correct=1)
Difference for with_signal/12784898_MSK AX.100 Mode 5.png:
  admin: 1 (correct=1)
  vadim: 0 (correct=0)
Difference for with_signal/12785650_GMSK.png:
  admin: 0 (correct=0)
  vadim: 1 (correct=1)
Difference for with_signal/12790094_GMSK.png:
  admin: 0 (correct=0)
  vadim: 1 (correct=1)
Difference for with_signal/12792119_GFSK.png:
  admin: 0 (correct=0)
  vadim: 1 (correct=1)
Difference for with_signal/12792120_GFSK.png:
  admin: 0 (correct=0)
  vadim: 1 (correct=1)
Difference for with_signal/12804109_FSK AX.25 G3RUH.png:
  guest: 1 (correct=1)
  admin: 0 (correct=0)
Difference for with_signal/12804535_CW.png:
  admin: 0 (correct=0)
  vadim: 1 (correct=1)
Difference for with_signal/12808083_BPSK.png:
  admin: 0 

In [21]:
data_path = "/Users/tedvtorov/Desktop/data_sample" # has 2 folders, with signal and without signal
# The sqlite database contains the properly sorted files. I will need to sort this dataset according to the database.
sorted_data_path = "/Users/tedvtorov/Desktop/data_sample_sorted"
# we first need to settle the differences though, so we will move the files to a separate folder and then we will decide what to do with them
differences_path = "/Users/tedvtorov/Desktop/data_sample_sorted/differences"

os.makedirs(differences_path, exist_ok=True)
os.makedirs(sorted_data_path, exist_ok=True)
os.makedirs(os.path.join(sorted_data_path, "with_signal"), exist_ok=True)
os.makedirs(os.path.join(sorted_data_path, "without_signal"), exist_ok=True)
os.makedirs(os.path.join(differences_path, "with_signal"), exist_ok=True)
os.makedirs(os.path.join(differences_path, "without_signal"), exist_ok=True)
for filename, correct, user1, decision1, user2, decision2 in differences:
    src = os.path.join(data_path, filename)
    dst = os.path.join(differences_path, filename)
    shutil.copy(src, dst)

# now we move on to sorting the rest of the files according to the database
for row in cur.execute("SELECT * FROM decisions ORDER BY filename"):
    id_, username, filename, decision, ground_truth, correct, timestamp = row
    if all(filename != diff[0] for diff in differences):
        src = os.path.join(data_path, filename)
        filename = filename.lstrip('with_signal/')
        filename = filename.lstrip('without_signal/')
        if decision:
            dst = os.path.join(sorted_data_path, "with_signal", filename)
        else:
            dst = os.path.join(sorted_data_path, "without_signal", filename)
        shutil.copy(src, dst)


In [22]:
# scale all the images in the completed dataset by a factor of 0.5. 
# use the PIL library to do this. We will do this for both the sorted dataset and the differences dataset, so that they are consistent.
from PIL import Image
def scale_image(image_path, scale_factor):
    with Image.open(image_path) as img:
        new_size = (int(img.width * scale_factor), int(img.height * scale_factor))
        img_resized = img.resize(new_size, Image.LANCZOS)
        img_resized.save(image_path)

for root, dirs, files in os.walk(sorted_data_path):
    for file in files:
        if file.endswith(".png") or file.endswith(".jpg") or file.endswith(".jpeg"):
            image_path = os.path.join(root, file)
            scale_image(image_path, 0.5)